In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import re
import ast
import csv
import json
import dotenv
from loguru import logger
from datetime import datetime
from typing import List, Dict
from datahub.ingestion.graph.client import DataHubGraph
from datahub_integrations.gen_ai.description_v2 import (
    extract_metadata_for_urn,
    transform_table_info_for_llm,
    call_bedrock_llm,
    DESCRIPTION_GENERATION_MODEL,
)
from datahub_integrations.gen_ai.term_suggestion_v2_context import (
    get_glossary_term_names_and_definition,
)
from helper import create_datahub_graph, write_llm_output_to_csv

In [ ]:
dotenv.load_dotenv("description_comparison/.env")

In [ ]:
# "glossary_term_id": "urn:li:glossaryTerm:Adoption.ReturnRate",
#             "term_name": "Return Rate",
#             "reasoning": "The table 'pet_profiles' likely contains information related to pet adoptions, and the 'Return Rate' glossary term is relevant to understanding the percentage of adopted animals that were returned to the shelter.",
#             "confidence_score": 8

In [ ]:
urns_dict = {
    "longtailcompanions": [
        "urn:li:dataset:(urn:li:dataPlatform:dbt,long_tail_companions.adoption.adoptions,PROD)",
        "urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.adoption.pet_profiles,PROD)",
        "urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.analytics.pet_details,PROD)",
        "urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.adoption.human_profiles,PROD)",
        "urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.adoption.pets,PROD)",
        "urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.analytics.active_customer_ltv,PROD)",
        "urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.analytics.dog_breeds,PROD)",
        "urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.ecommerce.items,PROD)",
        "urn:li:dataset:(urn:li:dataPlatform:looker,analytics.explore.retention_cohort,PROD)",  # <- no upstreams
        "urn:li:dataset:(urn:li:dataPlatform:looker,long_tail_companions.view.pet_details,PROD)",
        "urn:li:dataset:(urn:li:dataPlatform:looker,long_tail_companions.explore.monthly_adoption_projection,PROD)",
    ]
}
print(len(urns_dict))

In [ ]:
_GLOSSARY_AGGREGATION_QUERY = """
  {
    aggregateAcrossEntities(
      input: {query: "*", facets: ["glossaryTerms", "editedFieldGlossaryTerms"], types: [DATASET, DASHBOARD, DATA_FLOW]}
    ) {
      facets {
        field
        displayName
        aggregations {
          count
          value
          entity {
            urn
            type
          }
        }
      }
    }
  }
"""


def get_top_n_glossary_terms(
    graph_client: DataHubGraph, query: str, n: int = 20
) -> List[str]:
    query_output = graph_client.execute_graphql(query)
    glossary_terms_and_counts = {}
    for facet in query_output["aggregateAcrossEntities"]["facets"]:
        for aggregation in facet["aggregations"]:
            glossary_term_urn = aggregation.get("value")
            glossary_term_count = aggregation.get("count")
            glossary_terms_and_counts[glossary_term_urn] = (
                glossary_terms_and_counts.get(glossary_term_urn, 0)
                + glossary_term_count
            )
    top_n_glossary_terms = list(
        dict(
            sorted(
                glossary_terms_and_counts.items(),
                key=lambda item: item[1],
                reverse=True,
            )[:n]
        ).keys()
    )
    return top_n_glossary_terms


def parse_llm_output(text: str) -> tuple[str | None, Dict[str, str] | None]:
    match = re.search(r"\{[^}]*\}", text, re.DOTALL)
    if match:
        dict_str = match.group(0)
        # dict_str_cleaned = dict_str.replace("\n", " ").strip()
        dict_str_cleaned = dict_str.strip()
        dict_str_cleaned = re.sub("(?<=[a-z])'(?=[a-z])", "\\'", dict_str_cleaned)
        try:
            extracted_dict: dict = ast.literal_eval(dict_str_cleaned)
            table_terms: str = extracted_dict.pop("table_glossary_terms")
            table_terms = table_terms.strip("'\"").strip()
            return table_terms, extracted_dict

        except (SyntaxError, ValueError) as e:
            logger.info(f"Error evaluating dictionary: {e}. Text: {text}")
            return None, None
    else:
        logger.info("No dictionary found in the text.")
        return None, None


_TABLE_INFO_FOR_PROMPT = ["name", "description"]
_COLUMN_INFO_FOR_PROMPT = [
    "column_name",
    "descriptions",
    "sample_values",
    "nativeDataType",
]

In [ ]:
def generate_prompt(
    table_info: dict, column_info: dict, glossary_terms_info: dict
) -> str:
    prompt = f"""
        You are provided with metadata information about a table, its columns, and a list of glossary terms. Your task is to recommend up to three relevant glossary terms for the table and each column based solely on the provided metadata. 

        **Table Information:**
        {table_info}

        This is a python dictionary structured as follows:
        {{
            "name": "<name of the table>",
            "description": "<description of the table>"
        }}


        **Columns Information:**
        {column_info}

        This is a python dictionary where the keys are the column names and the values are dictionaries containing metadata:
        {{
            "<column_name1>": {{
                "column_name": "<name of the column 1>",
                "description": "<description of the column 1>",
                "sample_values": "<sample values of the column 1>",
                "nativeDataType": "<datatype of the column 1>"
            }},
            ...
        }}


        **Glossary Terms Information:**
        {glossary_terms_info}

        This is a python dictionary where the keys are unique glossary term identifiers, and the values are dictionaries containing metadata:
        {{
            "<glossary_term1_id>": {{
                "term_name": "<glossary term1 name>",
                "term_definition": "<description of the glossary term1>",
                "parent_node_name": "<parent node name if any>",
                "parent_node_definition": "<description of a parent node of the glossary term1>"
            }},
            ...
        }}

        **Task:**
        1. **Strictly** use the glossary terms provided in the "Glossary Terms Information" section. Do not introduce any new terms or use terms not explicitly listed.
        2. For each table and column, recommend up to three relevant glossary terms based on the relevance between the table/column metadata and glossary term metadata.
        3. Use only the column names provided in the "Columns Information" section when listing recommendations. Do not introduce or modify column names.
        4. Provide the final output in the following dictionary format:

        {{
            "table": [list of recommended glossary terms for table],
            "<column_name1>": [list of recommended glossary terms for column1],
            "<column_name2>": [list of recommended glossary terms for column2],
            ...
        }}

        5. Each recommended glossary term should include:
        {{
            "glossary_term_id": "<unique identifier of the glossary term>",
            "term_name": "<name of the glossary term>",
            "reasoning": "<brief explanation of why the term was recommended within 20 words, focusing on its relevance to the table/column details>",
            "confidence_score": <confidence score on a scale from 1 to 10, where 10 indicates a perfect match>
        }}

        **Important:** 
        - **Do not** generate or include any glossary terms, column names, or any other details that are not provided in the prompt.
        - Ensure that the final output dictionary is properly formatted and parsable.

    """
    return prompt


def write_llm_output_to_csv(llm_response: dict, csv_path: str = "") -> None:
    if csv_path == "":
        csv_path = (
            f"term_suggestion_output_{datetime.now().strftime('%m-%d-%Y_%H-%M-%S')}.csv"
        )
    with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
        csvwriter = csv.writer(csvfile)

        if len(llm_response[0]) == 4:
            csvwriter.writerow(
                ["instance", "urn", "table_glossary_terms", "column_glossary_terms"]
            )
        else:
            csvwriter.writerow(["instance", "urn", "raw_output"])

        for row in llm_response:
            row = list(row)
            if len(row) == 4:
                row[3] = json.dumps(row[3], indent=2)
            csvwriter.writerow(row)
    print(f"csv file {csv_path} created successfully")

In [ ]:
glossary_terms_suggestions = []
raw_llm_responses = []
for instance, entity_urns in urns_dict.items():
    graph_client = create_datahub_graph(instance)
    popular_glossary_terms = get_top_n_glossary_terms(
        graph_client, _GLOSSARY_AGGREGATION_QUERY, 20
    )
    glossary_terms_info = get_glossary_term_names_and_definition(
        popular_glossary_terms, graph_client
    )
    for urn in entity_urns:
        print(urn)
        entity = graph_client.get_entity_semityped(urn)
        extracted_entity_info = extract_metadata_for_urn(entity, urn, graph_client)
        table_info, column_info = transform_table_info_for_llm(extracted_entity_info)
        table_info_new = {
            k: v for k, v in table_info.items() if k in _TABLE_INFO_FOR_PROMPT
        }

        column_info_new = {}
        for column_name, column_info_dict in column_info.items():
            column_info_new[column_name] = {
                k: v
                for k, v in column_info_dict.items()
                if k in _COLUMN_INFO_FOR_PROMPT
            }

        prompt = generate_prompt(table_info, column_info, glossary_terms_info)
        raw_llm_response = call_bedrock_llm(
            prompt, model=DESCRIPTION_GENERATION_MODEL, max_tokens=4000
        )
        raw_llm_responses.append([urn, instance, raw_llm_response])
        table_terms, column_terms = parse_llm_output(raw_llm_response)
        glossary_terms_suggestions.append([urn, instance, table_terms, column_terms])
write_llm_output_to_csv(glossary_terms_suggestions)

In [ ]:
# parse_llm_output(raw_llm_response)
def parse_llm_output(text: str) -> tuple[str | None, Dict[str, str] | None]:
    match = re.search(r"\{[\s\S]*\}", text, re.DOTALL)
    # print(match)
    if match:
        dict_str = match.group(0)
        # dict_str_cleaned = dict_str.replace("\n", " ").strip()
        dict_str_cleaned = dict_str.strip()
        dict_str_cleaned = re.sub("(?<=[a-z])'(?=[a-z])", "\\'", dict_str_cleaned)
        try:
            extracted_dict: dict = ast.literal_eval(dict_str_cleaned)
            table_terms: List[str] = extracted_dict.pop("table")
            table_terms = table_terms.strip("'\"").strip()
            return table_terms, extracted_dict

        except (SyntaxError, ValueError) as e:
            logger.info(f"Error evaluating dictionary: {e}")  # . Text: {text}")
            return None, None
    else:
        logger.info("No dictionary found in the text.")
        return None, None


parse_llm_output(raw_llm_response)

In [ ]:
print(re.search(r"\{[^}]*\}", raw_llm_response, re.DOTALL).group(0))

In [ ]:
dict_str_cleaned = (
    re.search(r"\{[\s\S]*\}", raw_llm_response, re.DOTALL).group(0).strip()
)
print(re.sub("(?<=[a-z])'(?=[a-z])", "\\'", dict_str_cleaned))

In [ ]:
print(raw_llm_response)

In [ ]:
# glossary_terms_info_result
# details = parse_obj_as(List[TagSuggestion], details_raw)

In [ ]:
for urn in longtail_urns_list:
    print(urn)
    entity = graph_client.get_entity_semityped(urn)
    extracted_entity_info = extract_metadata_for_urn(entity, urn, graph_client)
    table_info, column_info = transform_table_info_for_llm(extracted_entity_info)

    table_info_for_prompt = ["name", "description"]
    column_info_for_prompt = [
        "column_name",
        "descriptions",
        "sample_values",
        "nativeDataType",
    ]

    table_info_new = {k: v for k, v in table_info.items() if k in table_info_for_prompt}

    column_info_new = {}
    for column_name, column_info_dict in column_info.items():
        column_info_new[column_name] = {
            k: v for k, v in column_info_dict.items() if k in column_info_for_prompt
        }

    prompt2 = f"""
        You are provided with metadata information about a table, its columns, and a list of glossary terms. Your task is to recommend up to three relevant glossary terms for the table and each column based solely on the provided metadata. 

        **Table Information:**
        {table_info}

        This is a python dictionary structured as follows:
        {{
            "name": "<name of the table>",
            "description": "<description of the table>"
        }}


        **Columns Information:**
        {column_info}

        This is a python dictionary where the keys are the column names and the values are dictionaries containing metadata:
        {{
            "<column_name1>": {{
                "column_name": "<name of the column 1>",
                "description": "<description of the column 1>",
                "sample_values": "<sample values of the column 1>",
                "nativeDataType": "<datatype of the column 1>"
            }},
            ...
        }}


        **Glossary Terms Information:**
        {glossary_terms_info}

        This is a python dictionary where the keys are unique glossary term identifiers, and the values are dictionaries containing metadata:
        {{
            "<glossary_term1_id>": {{
                "term_name": "<glossary term1 name>",
                "term_definition": "<description of the glossary term1>",
                "parent_node_name": "<parent node name if any>",
                "parent_node_definition": "<description of a parent node of the glossary term1>"
            }},
            ...
        }}

        **Task:**
        1. **Strictly** use the glossary terms provided in the "Glossary Terms Information" section. Do not introduce any new terms or use terms not explicitly listed.
        2. For each table and column, recommend up to three relevant glossary terms based on the relevance between the table/column metadata and glossary term metadata.
        3. Use only the column names provided in the "Columns Information" section when listing recommendations. Do not introduce or modify column names.
        4. Provide the final output in the following dictionary format:

        {{
            "table": [list of recommended glossary terms for table],
            "<column_name1>": [list of recommended glossary terms for column1],
            "<column_name2>": [list of recommended glossary terms for column2],
            ...
        }}

        5. Each recommended glossary term should include:
        {{
            "glossary_term_id": "<unique identifier of the glossary term>",
            "term_name": "<name of the glossary term>",
            "reasoning": "<brief explanation of why the term was recommended, focusing on its relevance to the table/column details>",
            "confidence_score": <confidence score on a scale from 1 to 10, where 10 indicates a perfect match>
        }}

        **Important:** 
        - **Do not** generate or include any glossary terms, column names, or any other details that are not provided in the prompt.
        - Ensure that the final output dictionary is properly formatted and parsable.

    """

In [ ]:
print(extracted_entity_info.keys())
print("==============")
print(extracted_entity_info["column_names"].values())
print("===================")
print(extracted_entity_info["column_sample_values"])
print("===================")
print(table_info)
print("===================")
print(column_info)
print("===================")
print(column_info.keys())

In [ ]:
table_info_for_prompt = ["name", "description"]
column_info_for_prompt = [
    "column_name",
    "descriptions",
    "sample_values",
    "nativeDataType",
]

table_info_new = {k: v for k, v in table_info.items() if k in table_info_for_prompt}
print(table_info_new)

print("=============================")

column_info_new = {}
for column_name, column_info_dict in column_info.items():
    column_info_new[column_name] = {
        k: v for k, v in column_info_dict.items() if k in column_info_for_prompt
    }
print(column_info_new)

In [ ]:
table_info = table_info_new
column_info = column_info_new
glossary_terms_info = glossary_terms_info_result

In [ ]:
prompt2 = f"""
You are provided with metadata information about a table, its columns, and a list of glossary terms. Your task is to recommend up to three relevant glossary terms for the table and each column based solely on the provided metadata. 

**Table Information:**
{table_info}

This is a python dictionary structured as follows:
{{
    "name": "<name of the table>",
    "description": "<description of the table>"
}}


**Columns Information:**
{column_info}

This is a python dictionary where the keys are the column names and the values are dictionaries containing metadata:
{{
    "<column_name1>": {{
        "column_name": "<name of the column 1>",
        "description": "<description of the column 1>",
        "sample_values": "<sample values of the column 1>",
        "nativeDataType": "<datatype of the column 1>"
    }},
    ...
}}


**Glossary Terms Information:**
{glossary_terms_info}

This is a python dictionary where the keys are unique glossary term identifiers, and the values are dictionaries containing metadata:
{{
    "<glossary_term1_id>": {{
        "term_name": "<glossary term1 name>",
        "term_definition": "<description of the glossary term1>",
        "parent_node_name": "<parent node name if any>",
        "parent_node_definition": "<description of a parent node of the glossary term1>"
    }},
    ...
}}

**Task:**
1. **Strictly** use the glossary terms provided in the "Glossary Terms Information" section. Do not introduce any new terms or use terms not explicitly listed.
2. For each table and column, recommend up to three relevant glossary terms based on the relevance between the table/column metadata and glossary term metadata.
3. Use only the column names provided in the "Columns Information" section when listing recommendations. Do not introduce or modify column names.
4. Provide the final output in the following dictionary format:

{{
    "table_glossary_terms": [list of recommended glossary terms for table],
    "<column_name1>": [list of recommended glossary terms for column1],
    "<column_name2>": [list of recommended glossary terms for column2],
    ...
}}

5. Each recommended glossary term should include:
{{
    "glossary_term_id": "<unique identifier of the glossary term>",
    "term_name": "<name of the glossary term>",
    "reasoning": "<brief explanation of why the term was recommended, focusing on its relevance to the table/column details>",
    "confidence_score": <confidence score on a scale from 1 to 10, where 10 indicates a perfect match>
}}

**Important:** 
- **Do not** generate or include any glossary terms, column names, or any other details that are not provided in the prompt.
- Ensure that the final output dictionary is properly formatted and parsable.

"""

llm_response2 = call_bedrock_llm(prompt2, max_tokens=4000)

In [ ]:
# print(prompt2)

In [ ]:
llm_response2 = call_bedrock_llm(prompt2, max_tokens=4000)

In [ ]:
print(llm_response2)